In [ ]:
# dataset link: https://www.kaggle.com/datasets/kinozyne/5-classes-of-fruits-dataset-train-test

In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import numpy as np
import matplotlib.pyplot as plt

## 1. Download and Prepare Dataset

In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kinozyne/5-classes-of-fruits-dataset-train-test")

print("Path to dataset files:", path)

Using Colab cache for faster access to the '5-classes-of-fruits-dataset-train-test' dataset.
Path to dataset files: /kaggle/input/5-classes-of-fruits-dataset-train-test


In [16]:
import os

TRAIN_DIR = f"{path}/dataset/train"
TEST_DIR = f"{path}/dataset/test"

# Check if directories exist
if not os.path.exists(TRAIN_DIR):
    print(f"Error: Train directory not found at {TRAIN_DIR}")
elif not os.path.exists(TEST_DIR):
    print(f"Error: Test directory not found at {TEST_DIR}")
else:
    print(f"Train directory: {TRAIN_DIR}")
    print(f"Test directory: {TEST_DIR}")

Train directory: /kaggle/input/5-classes-of-fruits-dataset-train-test/dataset/train
Test directory: /kaggle/input/5-classes-of-fruits-dataset-train-test/dataset/test


## 2. Data Augmentation and Data Generators

In [17]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

# Prepare data generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(192, 192),
    batch_size=32,
    class_mode='categorical')

print("Data generators prepared.")

Found 150 images belonging to 5 classes.
Found 50 images belonging to 5 classes.
Data generators prepared.


## 3. Build the CNN Model

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(192, 192, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                loss='categorical_crossentropy', # Use categorical_crossentropy for one-hot encoded labels
                metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 190, 190, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 95, 95, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 93, 93, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 46, 46, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 44, 44, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     7,929,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,023,877 (30.61 MB)

 Trainable params: 8,023,877 (30.61 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Training the Model

In [21]:
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
import math

# Define the Early Stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Define a learning rate function
def exponential_decay(epoch, learning_rate):
    initial_lrate = 0.001
    k = 0.1
    lrate = initial_lrate * math.exp(-k*epoch)
    return lrate

# Create the learning rate scheduler callback
lr_scheduler = LearningRateScheduler(exponential_decay)

# Train the model with the callback
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=60,
    callbacks=[early_stopping, lr_scheduler] # Add the callbacks here
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.1534 - loss: 2.9331 - learning_rate: 0.0010
Epoch 2/60


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.1250 - loss: 1.6710 - learning_rate: 9.0484e-04
Epoch 3/60


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.2710 - loss: 1.6140 - learning_rate: 8.1873e-04
Epoch 4/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4062 - loss: 1.4709 - learning_rate: 7.4082e-04
Epoch 5/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.3210 - loss: 1.4607 - learning_rate: 6.7032e-04
Epoch 6/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4688 - loss: 1.2627 - learning_rate: 6.0653e-04
Epoch 7/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.5296 - loss: 1.1905 - learning_rate: 5.4881e-04
Epoch 8/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3438 - loss: 1.3506 - learning_rate: 4.9659e-04
Epoch 9/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.5025 - loss: 1.0545 - learning_rate: 4.4933e-04
Epoch 10/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4375 - loss: 1.0293 - learning_rate: 4.0657e-04
Epoch 11/60
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.5469 - loss: 1.0769 - learning_rate: 3.6788e-04
Epoch 12/60
4/4 ━━━━━━━

## 5. Evaluate the Model

In [22]:
model.evaluate(test_generator)

2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6958 - loss: 0.7873


[0.7776464223861694, 0.699999988079071]

## 6. Save the Trained Model

In [23]:
model.save("my_cnn_model.keras")
print("Model saved successfully!")

Model saved successfully!


## 7. Load and Predict with the Model

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the saved model
loaded_model = load_model("my_cnn_model.keras") # Load the model saved in this notebook

image_paths_to_predict = ['/kaggle/input/5-classes-of-fruits-dataset-train-test/dataset/test/apple fruit/Image_36.jpg'] # Corrected filename capitalization

for image_path_to_predict in image_paths_to_predict:
    print(f"Predicting on image: {image_path_to_predict}")

    # Load and preprocess the image
    img = image.load_img(image_path_to_predict, target_size=(192, 192))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Rescale the image

    # Make a prediction
    predictions = loaded_model.predict(img_array)

    # Get the class labels from the test generator
    class_labels = list(test_generator.class_indices.keys())

    # Display the results
    print("\nPrediction probabilities:")
    for i, probability in enumerate(predictions[0]):
        print(f"{class_labels[i]}: {probability:.4f} ({probability*100:.2f}%)")

    predicted_class_index = np.argmax(predictions)
    predicted_class_label = class_labels[predicted_class_index]
    print(f"\nPredicted class: {predicted_class_label}")
    print("-" * 30)

Predicting on image: /kaggle/input/5-classes-of-fruits-dataset-train-test/dataset/test/apple fruit/Image_36.jpg


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 678ms/step

Prediction probabilities:
apple fruit: 0.6112 (61.12%)
banana fruit: 0.0093 (0.93%)
cherry fruit: 0.0574 (5.74%)
orange fruit: 0.0221 (2.21%)
strawberry fruiit: 0.2999 (29.99%)

Predicted class: apple fruit
------------------------------
